In [8]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import speech_recognition as sr

for index, name in enumerate(sr.Microphone.list_microphone_names()):
    print(index, name)

0 Microsoft Sound Mapper - Input
1 Microphone Array (AMD Audio Dev
2 Microsoft Sound Mapper - Output
3 Headphones (Realtek(R) Audio)
4 Speakers (Realtek(R) Audio)
5 Primary Sound Capture Driver
6 Microphone Array (AMD Audio Device)
7 Primary Sound Driver
8 Headphones (Realtek(R) Audio)
9 Speakers (Realtek(R) Audio)
10 Headphones (Realtek(R) Audio)
11 Speakers (Realtek(R) Audio)
12 Microphone Array (AMD Audio Device)
13 Speakers 1 (Realtek HD Audio output with HAP)
14 Speakers 2 (Realtek HD Audio output with HAP)
15 PC Speaker (Realtek HD Audio output with HAP)
16 Microphone (Realtek HD Audio Mic input)
17 Headphones (Realtek HD Audio 2nd output)
18 Stereo Mix (Realtek HD Audio Stereo input)
19 Microphone Array (AMDAfdInstall Wave Microphone - 0)


In [13]:
import os
import speech_recognition as sr  
import pyttsx3 
import gymnasium as gym  
import time
import keyboard  


engine = pyttsx3.init() 
recognizer = sr.Recognizer() 


mic_index = 1


env = gym.make("MountainCar-v0", render_mode="human")


COMMANDS = {
    "avance": 2, 
    "recule": 0,  
    "stop": 1  
}


def speak(text):
    """Le robot parle."""
    engine.say(text)
    engine.runAndWait()


def recognize_speech():
    """Utilise uniquement Google Speech pour la reconnaissance vocale."""
    with sr.Microphone(device_index=mic_index) as source:
        print("🎤 Parlez maintenant...")
        recognizer.adjust_for_ambient_noise(source, duration=1)  
        audio = recognizer.listen(source)

    try:
        command = recognizer.recognize_google(audio, language="fr-FR")
        print(f"✅ Reconnu : {command}")
        return command.lower()
    except sr.UnknownValueError:
        print("❌ Je n’ai pas compris.")
    except sr.RequestError:
        print("⚠️ Erreur de connexion.")

    return None


def confirm_command(command):
    """Demande une confirmation avant d'exécuter l'action."""
    speak(f"As-tu dit {command}? Réponds par oui ou non.")
    confirmation = recognize_speech()

    return confirmation == "oui"


obs, _ = env.reset()

while True:
    speak("Donnez une commande : avance, recule ou stop")
    command = recognize_speech()

    if command is None:
        continue  

    
    if keyboard.is_pressed("q"):  
        speak("Le robot s'arrête.")
        env.close()
        break  

    if "stop" in command:
        speak("Le robot s'arrête.")
        env.close()
        break  

    action = COMMANDS.get(command, None)

    if action is None:
        speak("Commande non reconnue. Essayez : avance, recule ou stop.")
        continue  

    print(f"🟡 Action détectée : {action}")  

    if confirm_command(command):  
        for _ in range(20):  
            obs, _, done, _, _ = env.step(action)
            env.render()
            time.sleep(0.05)

            if done: 
                obs, _ = env.reset()

        speak(f"Le robot {command}.")
    else:
        speak("D'accord, essaie encore.")


🎤 Parlez maintenant...
❌ Je n’ai pas compris.
🎤 Parlez maintenant...
✅ Reconnu : avance
🟡 Action détectée : 2
🎤 Parlez maintenant...
✅ Reconnu : recule
🎤 Parlez maintenant...
✅ Reconnu : avance
🟡 Action détectée : 2
🎤 Parlez maintenant...
✅ Reconnu : avance
🎤 Parlez maintenant...
❌ Je n’ai pas compris.
🎤 Parlez maintenant...
❌ Je n’ai pas compris.
🎤 Parlez maintenant...
✅ Reconnu : stoppe


In [14]:
import cv2
print("OpenCV version :", cv2.__version__)

OpenCV version : 4.11.0


In [ ]:
import cv2
import mediapipe as mp
import mediapipe.python.solutions.hands as mp_hands
import mediapipe.python.solutions.drawing_utils as mp_drawing
import gymnasium as gym
import highway_env
import numpy as np
import pygame
import time
import os

# =========================
# Initialize pygame sound
# =========================
pygame.mixer.init()

# Sound file path
sound_path = "acceleration.wav"

# Load sound
if os.path.exists(sound_path):
    acceleration_sound = pygame.mixer.Sound(sound_path)
else:
    acceleration_sound = None
    print(f"❌ Sound file not found: {sound_path}")

# =========================
# MediaPipe Hands
# =========================
hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

# =========================
# OpenCV Camera
# =========================
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("❌ Cannot access webcam")
    exit()

# =========================
# Highway Environment
# =========================
env = gym.make("highway-v0", render_mode="human")

obs, info = env.reset()

# =========================
# Finger detection function
# =========================
def count_fingers(hand_landmarks):

    finger_tips = [4, 8, 12, 16, 20]
    fingers = []

    # Thumb
    if hand_landmarks.landmark[finger_tips[0]].x < hand_landmarks.landmark[finger_tips[0] - 1].x:
        fingers.append(1)
    else:
        fingers.append(0)

    # Other fingers
    for tip in finger_tips[1:]:
        if hand_landmarks.landmark[tip].y < hand_landmarks.landmark[tip - 2].y:
            fingers.append(1)
        else:
            fingers.append(0)

    return fingers.count(1)

# =========================
# Main Loop
# =========================
try:
    while True:

        ret, frame = cap.read()

        if not ret:
            print("❌ Failed to capture frame")
            break

        # Flip image
        frame = cv2.flip(frame, 1)

        # Convert to RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Process hands
        results = hands.process(rgb_frame)

        action = 1  # Default action = IDLE

        if results.multi_hand_landmarks:

            for hand_landmarks in results.multi_hand_landmarks:

                # Draw landmarks
                mp_drawing.draw_landmarks(
                    frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS
                )

                # Count fingers
                fingers = count_fingers(hand_landmarks)

                # =========================
                # Gesture Control
                # =========================

                # 0 fingers = LEFT
                if fingers == 0:
                    action = 0
                    text = "LEFT"

                # 5 fingers = RIGHT
                elif fingers == 5:
                    action = 2
                    text = "RIGHT"

                    # Play acceleration sound
                    if acceleration_sound:
                        acceleration_sound.play()

                # 2-4 fingers = FASTER
                elif fingers >= 2:
                    action = 3
                    text = "FASTER"

                # 1 finger = SLOWER
                else:
                    action = 4
                    text = "SLOWER"

                # Display gesture
                cv2.putText(
                    frame,
                    f"Gesture: {text}",
                    (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 255, 0),
                    2
                )

        # =========================
        # Step environment
        # =========================
        obs, reward, terminated, truncated, info = env.step(action)

        env.render()

        # Reset if episode finished
        if terminated or truncated:
            obs, info = env.reset()

        # Show webcam
        cv2.imshow("Hand Gesture Control", frame)

        # Quit with Q
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

        time.sleep(0.03)

finally:

    cap.release()
    cv2.destroyAllWindows()
    env.close()